In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
import requests
import datetime
from selenium import webdriver
from time import sleep
import os
from urllib.parse import urljoin
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
import re

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'CL CMF' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running CL CMF Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

mainURL = 'https://www.cmfchile.cl/portal/principal/613/w3-propertyvalue-43336.html'

regdict={

         regulatorName + ' 1': 'entidades_valores',
         regulatorName + ' 2': 'entidades_seguros',
         regulatorName + ' 3': 'entidades_bancos',


        }


Typology={

       regulatorName + ' 1': 'Entidades fiscalizadas - Mercado de valores',
       regulatorName + ' 2': 'Entidades fiscalizadas - Mercado de seguros',
       regulatorName + ' 3': 'Entidades fiscalizadas - Bancos e Inst. Financieras',

        }

skip_by_reg = {
    f"{regulatorName} 1": [
        "Abogados Calificadores",
        "Deportivas no profesionales, de beneficencia o educacionales",
        'Emisores de Valores de Oferta Pública',
        'Fondos de Inversión No Rescatables',
        'Fondos de Inversión Rescatables',
        'Fondos Mutuos',
        'Fondos para la Vivienda',
        'Soc. Adm. Fondos Mutuos',
        'Soc. Adm. Fondos para la Vivienda',
        'Valores Extranjeros'
    ],
    f"{regulatorName} 2":[
        "Agentes de Ventas de Seguros", 
        "Cías. Reaseguradoras Grales Nacionales", 
        "Cías. Reaseguradoras Vida Nacionales", 
        "Corredores de Reaseguros Extranjeros", 
        "Corredores de Seguros - Persona Natural", 
        "Liquidadores de Siniestros - persona natural"
    ],
    f"{regulatorName} 3": [
        "Filiales de Bancos Chilenos en el Extranjero",
        "Oficinas de Representación de Bancos Chilenos en el Extranjero", 
        "Auditores Externos", 
        "Sucursales de Bancos Chilenos en el Extranjero Fiscalizados"
    ]}

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


def col_ci(df, name):
    col_map = {c.lower(): c for c in df.keys()}
    key = name.lower()
    if key not in col_map:
        raise KeyError(f"Column not found (case-insensitive): {name}")
    return df[col_map[key]]

def find_key_contains(obj, needle):
    keys = list(obj.keys()) if hasattr(obj, "keys") else list(obj.columns)
    needle = needle.lower()
    for k in keys:
        if needle in k.lower():
            return k
    return ""  # not found



In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36"
}

resp = requests.get(mainURL, headers=headers, timeout=30)
# If shows 443 code, change to the Chile Internet 
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# examples:
entidades_tabs = soup.find("div", id="entidades_tabs")
tab_links = soup.select("#entidades_tabs .nav-link")
url = 'https://www.cmfchile.cl/portal/principal/613/'

In [ ]:
for reg in regdict:
    if reg!= regulatorName+' 3':
        links = entidades_tabs.find('div',id=regdict[reg]).find_all('li')
        for link in links:
            if link.text in skip_by_reg[reg]:
                continue
            href = link.find('a')['href']
            print(link.text)
            topo = link.text
            
            detail_url = urljoin(url, href)
            print(detail_url)
            retry = Retry(
                total=5,
                connect=5,
                read=5,
                status=5,
                backoff_factor=1,  # 1s, 2s, 4s, 8s...
                status_forcelist=[404,429, 500, 502, 503, 504],
                allowed_methods=["GET"],
                raise_on_status=False
            )
            session = requests.Session()
            session.headers.update(headers)
            session.mount("https://", HTTPAdapter(max_retries=retry))
            session.mount("http://", HTTPAdapter(max_retries=retry))
            driver.get(detail_url)
            sleep(3)
            tab_content = driver.find_element(By.ID,'enti_fiscalizados')
            sleep(3)
            table = tab_content.find_element(By.TAG_NAME,'table')
            rows = table.find_elements(By.TAG_NAME, "tr")

            all_rows = []
            for row in rows:
                cells = row.find_elements(By.TAG_NAME, "td")
                if not cells:
                    continue
                row_data = []
                row_links = []

                for cell in cells:
                    row_data.append(cell.text.strip())
                    links = cell.find_elements(By.TAG_NAME, "a")
                    if links:
                        href = links[0].get_attribute("href")
                        row_links.append(urljoin(url, href))
                    else:
                        row_links.append("")

                all_rows.append({"text": row_data, "hrefs": row_links})

            for rows in all_rows:
                rut_ = rows['text'][0]
                company_name = rows['text'][1]
                print(company_name)
                sqldict['Name'].append(company_name)
                sqldict['InternalID_1'].append(rut_)
                sqldict['InternalID_1_type'].append('RUT')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict['RegulationType'].append('Regulated')
                sqldict['ListName'].append(Typology[reg])
                sqldict['Typology'].append(topo)
                #print(rows['hrefs'][0])


                retry = Retry(
                total=5,
                connect=5,
                read=5,
                status=5,
                backoff_factor=1,  # 1s, 2s, 4s, 8s...
                status_forcelist=[404,429, 500, 502, 503, 504],
                allowed_methods=["GET"],
                raise_on_status=False
                )
                
                session = requests.Session()
                session.mount("https://", HTTPAdapter(max_retries=retry))
                session.mount("http://", HTTPAdapter(max_retries=retry))
                company_detail = session.get(rows['hrefs'][0], headers=headers, timeout=30)
                
                company_detail = BeautifulSoup(company_detail.text, "html.parser")
                table_detail = company_detail.find('table')
                data = {}
                try:
                    sleep(3)
                    table_detail_rows = table_detail.find_all("tr")
                except:
                    sleep(3)
                    company_detail = session.get(rows['hrefs'][0], headers=headers, timeout=30)
                    company_detail = BeautifulSoup(company_detail.text, "html.parser")
                    table_detail = company_detail.find('table')
                    table_detail_rows = table_detail.find_all("tr")
                
                # company_detail = BeautifulSoup(company_detail.text, "html.parser")
                # table_detail = company_detail.find('table')                
                for row in table_detail_rows:
                    th = row.find("th")
                    td = row.find("td")
                    if not th or not td:
                        continue
                    label = th.get_text(strip=True)
                    value = td.get_text(" ", strip=True)
                    data[label] = value

                if data:

                    col_address = find_key_contains(data, "domicilio")
                    address_ = data[col_address] if col_address else "" 
                    
                    col_city = find_key_contains(data, "ciudad")
                    city_ = data[col_city] if col_city else ""  
                    try:
                        register_date = col_ci(data, "Fecha de Inscripción")
                    except:
                        register_date=''
                    try:
                        phone_ = col_ci(data, 'Teléfono')
                    except:
                        phone_=''
                    try:
                        fax_ = col_ci(data,'Fax')
                    except:
                        fax_ = ''

                    try:
                        email_ = col_ci(data,'e-mail de contacto')
                    except:
                        email_ = ''
                    try:
                        web_ = col_ci(data,'Sitio web')
                    except:
                        web_=''
                    try:
                        zip_ = col_ci(data,'Código Postal')
                    except:
                        zip_=''
                    sqldict['RegulationDate'].append(register_date)
                    sqldict['Phone'].append(phone_)
                    sqldict['Fax'].append(fax_)
                    sqldict['Address_1'].append(address_)
                    sqldict['City'].append(city_ if city_!='---' else '')
                    sqldict['Email'].append(email_)
                    sqldict['Website'].append(web_)
                    sqldict['Zip'].append(zip_)
                sqldict = bourange_same_length_array(sqldict)


    else :
        links = entidades_tabs.find('div',id=regdict[reg]).find_all('li')
        for link in links:
            if link.text in skip_by_reg[reg]:
                continue
            href = link.find('a')['href']
            print(link.text)
            topo = link.text
            detail_url = urljoin(url, href)
            print(detail_url)
            retry = Retry(
                total=5,
                connect=5,
                read=5,
                status=5,
                backoff_factor=1,  # 1s, 2s, 4s, 8s...
                status_forcelist=[404,429, 500, 502, 503, 504],
                allowed_methods=["GET"],
                raise_on_status=False
            )
            session = requests.Session()
            session.headers.update(headers)
            session.mount("https://", HTTPAdapter(max_retries=retry))
            session.mount("http://", HTTPAdapter(max_retries=retry))
            driver.get(detail_url)
            sleep(3)
            tab_content = driver.find_element(By.ID,'enti_fiscalizados')
            sleep(3)
            try:
                table = tab_content.find_element(By.TAG_NAME,'table')
                rows = table.find_elements(By.TAG_NAME, "tr")
            except:
                rows = ''

            all_rows = []
            if rows:
                for row in rows:
                    cells = row.find_elements(By.TAG_NAME, "td")
                    if not cells:
                        continue
                    row_data = []
                    row_links = []

                    for cell in cells:
                        row_data.append(cell.text.strip())
                        links = cell.find_elements(By.TAG_NAME, "a")
                        if links:
                            href = links[0].get_attribute("href")
                            row_links.append(urljoin(url, href))
                        else:
                            row_links.append("")

                    all_rows.append({"text": row_data, "hrefs": row_links})

                for rows in all_rows:
                    rut_ = rows['text'][0]
                    company_name = rows['text'][1]
                    print(company_name)
                    sqldict['Name'].append(company_name)
                    sqldict['InternalID_1'].append(rut_)
                    sqldict['InternalID_1_type'].append('RUT')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['Typology'].append(topo)
                    #print(rows['hrefs'][0])
                    retry = Retry(
                    total=5,
                    connect=5,
                    read=5,
                    status=5,
                    backoff_factor=1,  # 1s, 2s, 4s, 8s...
                    status_forcelist=[404,429, 500, 502, 503, 504],
                    allowed_methods=["GET"],
                    raise_on_status=False
                    )
                    
                    session = requests.Session()
                    session.mount("https://", HTTPAdapter(max_retries=retry))
                    session.mount("http://", HTTPAdapter(max_retries=retry))
                    company_detail = session.get(rows['hrefs'][0], headers=headers, timeout=30)
                    
                    company_detail = BeautifulSoup(company_detail.text, "html.parser")
                    table_detail = company_detail.find('table')
                    data = {}
                    try:
                        sleep(3)
                        table_detail_rows = table_detail.find_all("tr")
                    except:
                        sleep(3)
                        company_detail = session.get(rows['hrefs'][0], headers=headers, timeout=30)
                        company_detail = BeautifulSoup(company_detail.text, "html.parser")
                        table_detail = company_detail.find('table')
                        table_detail_rows = table_detail.find_all("tr")
                    
                    # company_detail = BeautifulSoup(company_detail.text, "html.parser")
                    # table_detail = company_detail.find('table')                
                    for row in table_detail_rows:
                        th = row.find("th")
                        td = row.find("td")
                        if not th or not td:
                            continue
                        label = th.get_text(strip=True)
                        value = td.get_text(" ", strip=True)
                        data[label] = value

                    if data:

                        col_address = find_key_contains(data, "domicilio")
                        address_ = data[col_address] if col_address else "" 
                        
                        col_city = find_key_contains(data, "ciudad")
                        city_ = data[col_city] if col_city else ""  
                        try:
                            register_date = col_ci(data, "Fecha de Inscripción")
                        except:
                            register_date=''
                        try:
                            phone_ = col_ci(data, 'Teléfono')
                        except:
                            phone_=''
                        try:
                            fax_ = col_ci(data,'Fax')
                        except:
                            fax_ = ''

                        try:
                            email_ = col_ci(data,'e-mail de contacto')
                        except:
                            email_ = ''
                        try:
                            web_ = col_ci(data,'Sitio web')
                        except:
                            web_=''
                        try:
                            zip_ = col_ci(data,'Código Postal')
                        except:
                            zip_=''
                        sqldict['RegulationDate'].append(register_date)
                        sqldict['Phone'].append(phone_)
                        sqldict['Fax'].append(fax_)
                        sqldict['Address_1'].append(address_)
                        sqldict['City'].append(city_ if city_!='---' else '')
                        sqldict['Email'].append(email_)
                        sqldict['Website'].append(web_)
                        sqldict['Zip'].append(zip_)
                    sqldict = bourange_same_length_array(sqldict)
            else:
                ULs = driver.find_element(By.ID, "article_i__cmf20_pa_entidadesBancos_listado_1")

                for li in ULs.find_elements(By.TAG_NAME, "li"):
                    if "Código " not in li.text:
                        continue
                    

                    uls = li.find_elements(By.TAG_NAME, "ul")
                    sub_text = uls[0].text if uls else ""
                    main_text = li.text.replace(sub_text, "").strip()
                    name = main_text.split("(", 1)[0].strip()
                    id_ = main_text.split("(", 1)[-1].split(':')[1].split('-')[0].strip()

                    if name:
                        print(name)
                        # print(id_)
                        sqldict['Name'].append(name)
                        sqldict['InternalID_1'].append(id_)
                        sqldict['InternalID_1_type'].append('Código SBIF')
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegCtry'].append(reg.split(' ')[0])
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['Typology'].append(topo)
                        sqldict = bourange_same_length_array(sqldict)

Administradoras de Cartera
https://www.cmfchile.cl/portal/principal/613/w3-propertyvalue-21506.html
ADMINISTRADORA GENERAL DE FONDOS SECURITY S.A.
ADMINISTRADORA GENERAL DE FONDOS SURA S.A.
ALZA ADMINISTRADORA GENERAL DE FONDOS S.A.
AMERIS CAPITAL ADMINISTRADORA GENERAL DE FONDOS S.A.
AMICORP ADMINISTRADORA GENERAL DE FONDOS S.A.
ASSET ADMINISTRADORA GENERAL DE FONDOS S.A.
AVANTE ADMINISTRADORA GENERAL DE FONDOS S.A.
AZIMUT INVESTMENTS S.A. ADMINISTRADORA GENERAL DE FONDOS
BANCHILE ADMINISTRADORA GENERAL DE FONDOS S.A.
BANCHILE CORREDORES DE BOLSA S.A.
BANCO INTERNACIONAL ADMINISTRADORA GENERAL DE FONDOS S.A.
BANCOESTADO S.A. ADMINISTRADORA GENERAL DE FONDOS
BANCOESTADO S.A. CORREDORES DE BOLSA
BCI ASSET MANAGEMENT ADMINISTRADORA GENERAL DE FONDOS S.A.
BCI CORREDOR DE BOLSA S.A.
BICE INVERSIONES ADMINISTRADORA GENERAL DE FONDOS S.A.
BICE INVERSIONES CORREDORES DE BOLSA S.A.
BTG PACTUAL CHILE S.A. ADMINISTRADORA GENERAL DE FONDOS
BTG PACTUAL CHILE S.A. CORREDORES DE BOLSA
CAPITAL ADVISO

In [ ]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
driver.quit()
sleep(3)

: 